In [1]:
import numpy as np
from torch import nn, torch
from torch.utils.data import DataLoader, TensorDataset


class DeepSVDDNetwork(nn.Module):
    def __init__(self, n_features: int, latent_dim: int = 8, hidden: tuple = (32, 16)):
        super().__init__()
        layers = []
        prev = n_features 
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h 
        layers.append(nn.Linear(prev, latent_dim))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def init_center_c(X_normal, model, device='cpu', eps=0.1):
    model.eval()
    X_arr = X_normal.values if hasattr(X_normal, 'values') else X_normal
    X_tensor = torch.tensor(X_arr, dtype=torch.float32).to(device)

    with torch.no_grad():
        z = model(X_tensor)
        c = torch.mean(z, dim=0)
        c[(c.abs() < eps)] = eps

    return c

def train_svdd(X_normal, n_features, epochs=100, batch_size=512, lr=1e-3,
            latent_dim=8, hidden=(32, 16), device='cpu', seed=131):
    torch.manual_seed(seed)
    model = DeepSVDDNetwork(n_features, latent_dim, hidden).to(device)
    c = init_center_c(X_normal, model, device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    X_arr = X_normal.values if hasattr(X_normal, 'values') else X_normal
    ds = TensorDataset(torch.tensor(X_arr, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for (xb,) in dl:
            xb = xb.to(device)
            opt.zero_grad()
            z = model(xb)
            loss = torch.mean(torch.sum((z - c)**2, dim=1))
            loss.backward()
            opt.step()

    model.eval()
    return model, c


@torch.no_grad()
def get_svdd_scores(model, c, X, device='cpu', batch_size=4096):
    model.eval()
    X_arr = X.values if hasattr(X, 'values') else X
    X_tensor = torch.tensor(X_arr, dtype=torch.float32)

    scores = np.zeros(len(X_tensor), dtype=np.float32)

    for start in range(0, len(X_tensor), batch_size):
        xb = X_tensor[start:start + batch_size].to(device)
        z = model(xb)

        err = torch.sum((z - c)** 2, dim=1).cpu().numpy()
        scores[start:start + batch_size] = err

    return scores 

In [ ]:
import sys

import optuna
import sympy
import sympy.logic
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y

sympy.logic = sys.modules['sympy.logic']

def objective_svdd(trial):
    hidden_str = trial.suggest_categorical('hidden', ['64_32', '128_64','256_128',])
    hidden = tuple(map(int, hidden_str.split('_')))

    params = {
        'latent_dim': trial.suggest_categorical('latent_dim', [2, 4, 8, 16, 32]),
        'hidden': hidden,
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [256, 512, 1024]),
        'epochs': 100
    }
    aucs = []
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
        qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

        X_tr_f = feature_engeneering(
            df=X_tr, qt=qt, train_median=train_median,
            train_mad=train_mad, tsh_quantile=tsh_q,
            fti_quantile=fti_q, tt4_quantile=tt4_q
        )
                    
        X_val_f = feature_engeneering(
            df=X_val, qt=qt, train_median=train_median,
            train_mad=train_mad, tsh_quantile=tsh_q,
            fti_quantile=fti_q, tt4_quantile=tt4_q
        )
    
        X_tr_normal = X_tr_f[y_tr == 0]
        
        scaler = StandardScaler()
        scaler.fit(X_tr_normal)
        
        X_tr_scaled = scaler.transform(X_tr_normal)
        X_val_scaled = scaler.transform(X_val_f)

        model, c = train_svdd(
            X_normal=X_tr_scaled,
            n_features=X_tr_scaled.shape[1],
            **params,
            device='cpu',
            seed=131 + fold
        )

        val_scores = get_svdd_scores(model, c, X_val_scaled, device='cpu')
        pr_auc = average_precision_score(y_val, val_scores)
        aucs.append(pr_auc)

        trial.report(float(np.mean(aucs)), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(aucs))

study_svdd = optuna.create_study(direction='maximize')
study_svdd.optimize(objective_svdd, n_trials=200)

print(f'Best_params: {study_svdd.best_params}')
print(f'Best value: {study_svdd.best_value}')

c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-22 16:41:55,007] A new study created in memory with name: no-name-78275d02-2f83-45e7-b94b-c22a408e6c77
[I 2026-11-22 16:42:02,408] Trial 0 finished with value: 0.4758210228774985 and parameters: {'hidden': '64_32', 'latent_dim': 4, 'lr': 0.00014171627755662445, 'batch_size': 256}. Best is trial 0 with value: 0.4758210228774985.
[I 2026-11-22 16:42:12,051] Trial 1 finished with value: 0.49896101376194385 and parameters: {'hidden': '64_32', 'latent_dim': 32, 'lr': 0.0015423823013679169, 'batch_size': 1024}. Best is trial 1 with value: 0.49896101376194385.
[I 2026-11-22 16:42:27,323] Trial 2 finished with value: 0.4670803630975128 and parameters: {'hidden': '256_128', 'latent_dim': 2, 'lr': 0.00018014031

Best_params: {'hidden': '64_32', 'latent_dim': 8, 'lr': 0.006649834394010357, 'batch_size': 256}
Best value: 0.7840796902866106


In [3]:
best_params = study_svdd.best_params
oof_svdd = np.zeros(len(X))
synth_svdd = []

hidden_str = best_params['hidden']
best_params['hidden'] = tuple(map(int, hidden_str.split('_')))

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q, 
        fti_quantile=fti_q, tt4_quantile=tt4_q 
    )
    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)

    model, c = train_svdd(
        X_normal=X_tr_scaled,
        n_features=X_tr_scaled.shape[1],
        **best_params,
        device='cpu'
    )

    oof_svdd[val_idx] = get_svdd_scores(model=model, c=c, X=X_val_scaled, device='cpu')
    synth_svdd.extend(get_svdd_scores(model=model, c=c, X=X_synth_scaled, device='cpu'))

np.save('oof_svdd.npy', oof_svdd)
np.save('synth_svdd.npy', synth_svdd)